In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import squidpy as sq
from scipy.ndimage import gaussian_filter
from skimage.color import label2rgb
from skimage.draw import ellipse
from skimage.measure import label, regionprops, regionprops_table
from skimage.transform import rotate
import warnings

from scipy.stats import kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from skimage.segmentation import expand_labels, watershed



In [ ]:
# This below is a huge sanity check to make sure that nimbus is fed the proper segmentation mask
# Identify the TMA once from the patches
TMA = sorted_key(top_patches_coords.iloc[0, 4])  # This is the organizer 

print("Loading:", TMA)

# Load mask once
cell_mask = next(f for f in cell_masks if TMA in os.path.basename(f))
img = cv2.imread(cell_mask, cv2.IMREAD_UNCHANGED)

# Load H&E once
HNE = next(f for f in HNE_aligned if TMA in os.path.basename(f))
full_HNE = cv2.imread(HNE, cv2.IMREAD_UNCHANGED)

# Load DAPI once
DAPI_path = next(f for f in DAPI if TMA in os.path.basename(f))
DAPI = cv2.imread(DAPI_path, cv2.IMREAD_UNCHANGED)

# Load OG DAPI once
OG_DAPI_path = next(f for f in OG_DAPI if TMA in os.path.basename(f))
OG_DAPI = np.load(OG_DAPI_path)

expanded_DAPI_path = next(f for f in expanded_DAPI if TMA in os.path.basename(f))
expanded_DAPI = np.load(expanded_DAPI_path)

print("OG DAPI:", OG_DAPI_path)
print("Expanded DAPI:", expanded_DAPI_path)

print("DAPI:", DAPI_path)
print("Mask:", cell_mask)
print("HNE:", HNE)
print("Mask shape:", img.shape)
print("HNE shape:", full_HNE.shape)
print("DAPI shape:", DAPI.shape)
print("OG DAPI shape:", OG_DAPI.shape)
print("Expanded DAPI shape:", expanded_DAPI.shape)


# Optional: visualize full mask once
plt.figure(figsize=(8,8))
plt.imshow(full_HNE)
plt.imshow(label2rgb(img, bg_label=0), alpha = 0.5)
plt.title(TMA)
plt.axis("off")
plt.show()

plt.figure(figsize=(8,8))
plt.imshow(expanded_DAPI)
plt.title(expanded_DAPI)
plt.axis("off")
plt.show()


# Now loop over patches only
for row in top_patches_coords.itertuples():

    rank = row[1]
    attention = row[2]
    x = row[3]
    y = row[4]
    patch_path = row[5]

    HE_patch = np.load(patch_path)

    blurred = cv2.GaussianBlur(HE_patch, (5, 5), 0)

    masks, flows, styles = model.eval(blurred)

    if masks.max() == 0:
        print(f"No cells in {patch_path}, skipping")
        continue

    n_cells = masks.max()

    # Crop the corresponding regions
    mask_crop = img[y:y+patch_size, x:x+patch_size]
    HNE_crop = full_HNE[y:y+patch_size, x:x+patch_size]
    DAPI_crop = DAPI[y:y+patch_size, x:x+patch_size]
    OG_DAPI_crop = OG_DAPI[y:y+patch_size, x:x+patch_size]
    Expanded_DAPI_crop = expanded_DAPI[y:y+patch_size, x:x+patch_size]


    fig, axes = plt.subplots(1, 7, figsize=(25,5))

    axes[0].imshow(HE_patch)
    axes[0].set_title("Stored H&E patch")
    axes[0].axis("off")


    axes[1].imshow(mask_crop)
    axes[1].imshow(HNE_crop)
    axes[1].imshow(label2rgb(mask_crop, bg_label=0), alpha=0.4)
    axes[1].set_title("Nimbus mask overlay")
    axes[1].axis("off")

    axes[2].imshow(label2rgb(masks, image=blurred, alpha=0.5, bg_label=0))
    axes[2].set_title(f"Cellpose ({n_cells} cells)")
    axes[2].axis("off")

    axes[3].imshow(label2rgb(mask_crop, bg_label=0))
    axes[3].set_title("Mask crop")
    axes[3].axis("off")

    axes[4].imshow(DAPI_crop)
    axes[4].set_title("DAPI crop")
    axes[4].axis("off")

    axes[5].imshow(OG_DAPI_crop)
    axes[5].set_title("OG DAPI crop")
    axes[5].axis("off")

    axes[6].imshow(Expanded_DAPI_crop)
    axes[6].set_title("OG DAPI crop")
    axes[6].axis("off")


    fig.suptitle(
        f"{TMA} | Rank {rank} | Attention {attention:.3f} | x={x}, y={y}",
        fontsize=13,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
properties = [  'label',
                'area',
                'area_bbox',
                'area_convex',
                'area_filled',
                'bbox',
                'centroid',
                'centroid_local',
                'centroid_weighted',
                'centroid_weighted_local',
                'equivalent_diameter_area',
                'extent',
                'feret_diameter_max',
                'num_pixels',
                'solidity',
                'axis_major_length',
                'axis_minor_length',
                'eccentricity',
                'orientation',
                'perimeter',
                'perimeter_crofton',
                'euler_number',
                'moments',
                'moments_central',
                'moments_hu',
                'moments_normalized',
                'moments_weighted',
                'moments_weighted_central',
                'moments_weighted_hu',
                'moments_weighted_normalized',
                'inertia_tensor',
                'inertia_tensor_eigvals',
                'intensity_max',
                'intensity_mean',
                'intensity_min',
                'intensity_std']

In [ ]:
#root_path = "/projects/ag-bozek/jshemons/top_patches"
root_path = "/projects/ag-bozek/jshemons/bot_patches"

all_results = {}

for tma_dir in os.listdir(root_path):
    tma_path = os.path.join(root_path, tma_dir)
    if not os.path.isdir(tma_path):
        continue

    patch_files = [f for f in os.listdir(tma_path) if f.endswith(".npy")]

    patches = []
    blurred_patches = []
    patch_names = []

    for patch_file in patch_files:
        patch_path = os.path.join(tma_path, patch_file)
        patch = np.load(patch_path, allow_pickle=True)

        patches.append(patch)
        blurred_patches.append(cv2.GaussianBlur(patch, (5, 5), 0))
        patch_names.append(patch_file)

    if len(blurred_patches) == 0:
        all_results[tma_dir] = pd.DataFrame()
        continue

    masks_list, flows, styles = model.eval(blurred_patches)

    rows = []

    for patch, patch_file, masks in zip(patches, patch_names, masks_list):
        if masks.max() == 0:
            print(f"No cells in {patch_file}")
            continue

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=RuntimeWarning)
            props = regionprops_table(
                masks,
                intensity_image=patch,
                properties=properties
            )

        props_df = pd.DataFrame(props)
        props_df["patch_name"] = patch_file
        props_df["tma"] = tma_dir
        props_df["n_cells_detected_in_patch"] = len(props_df)

        rows.append(props_df)

    tma_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    tma_df = tma_df.drop(columns=cols_to_drop, errors="ignore")

    all_results[tma_dir] = tma_df
    print(f"{tma_dir}: {len(tma_df)} cells detected")

master_df = pd.concat(all_results.values(), ignore_index=True)
master_df = master_df.drop(columns=cols_to_drop, errors="ignore")

print(f"\nTotal cells: {len(master_df)}")